# Dataset 2 – Human Gut Microbiome: Complete EDA

**Source:** [knights-lab/dietstudy_analyses](https://github.com/knights-lab/dietstudy_analyses)  
**Study:** Longitudinal dietary intervention — 37 participants × ~17 days  
**Arms:** MCT oil vs. Extra-Virgin Olive Oil (EVOO)  
**Data layers:** 16S rRNA microbiome · daily nutrition diaries · blood biomarkers

**Sections:** Data Loading → Cleaning → Study Design → Microbiome → Nutrition → Food Groups → Blood Tests → Diet↔Microbiome

## Imports

All libraries needed throughout the notebook.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import mannwhitneyu

sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams["figure.dpi"] = 120
BASE_DIR = "dietstudy_data"

✅ Libraries imported and plot style configured.

## Data Loading

We load all 7 distinct tables. `food_map`, `taxonomy_norm_map`, and `nutr_65` are subsets/reformats of existing data and are skipped.

In [ ]:
sep = '\t'

# Metadata
sample_map = pd.read_csv(f'{BASE_DIR}/maps/SampleID_map.txt', sep=sep)
sample_map.rename(columns={'#SampleID': 'SampleID'}, inplace=True)

user_map  = pd.read_csv(f'{BASE_DIR}/maps/UserName_map.txt', sep=sep)
user_long = pd.read_csv(f'{BASE_DIR}/maps/UserName_map_long.txt', sep=sep)

# Microbiome
burst = pd.read_csv(f'{BASE_DIR}/microbiome/BURST.tax.txt', sep=sep, index_col=0)
burst.index.name = 'taxonomy'

# Diet
nutrition = pd.read_csv(f'{BASE_DIR}/diet/nutrition_totals.txt', sep=sep)
nutrition.rename(columns={'X.SampleID': 'SampleID'}, inplace=True)
diet_fiber = pd.read_csv(f'{BASE_DIR}/diet/diet.fiber.txt', sep=sep, index_col=0)
food_tax   = pd.read_csv(f'{BASE_DIR}/diet/diet.taxonomy.txt', sep=sep)

tables = [('sample_map',sample_map),('user_map',user_map),('user_long',user_long),
          ('burst',burst),('nutrition',nutrition),('diet_fiber',diet_fiber),('food_tax',food_tax)]
print(f'  Table                Shape')
print('  ' + '-'*35)
for name, df in tables:
    print(f'  {name:<20} {str(df.shape)}')

✅ All 7 tables loaded:
- **Metadata:** 643 samples, 37 participants, 74 blood-draw records
- **Microbiome (BURST):** 4,583 taxa × 538 samples (raw counts)
- **Nutrition:** 580 sample-days × 103 nutrients
- **Diet fiber:** 1,452 food items × 581 samples
- **Food taxonomy:** 8,767 food items → hierarchical categories

## Data Cleaning

The `sample_map` contains 643 rows including blank lab controls and participant rows where no stool was collected (`fecal.status = 0`). We keep only confirmed fecal samples from study participants.

In [ ]:
fecal = sample_map[
    (sample_map['fecal.status'] == 1) &
    (sample_map['UserName'].str.startswith('MCTs', na=False))
].copy()

n_blanks   = (~sample_map['UserName'].str.startswith('MCTs', na=False)).sum()
n_no_stool = (sample_map['UserName'].str.startswith('MCTs', na=False) &
              (sample_map['fecal.status'] == 0)).sum()

print(f'Total rows in sample_map:          {len(sample_map)}')
print(f'  Blank/control rows removed:      {n_blanks}')
print(f'  fecal.status==0 (no stool):      {n_no_stool}')
print(f'Valid fecal samples retained:      {len(fecal)}')
print(f'Unique participants:               {fecal["UserName"].nunique()}')
print(f'Study days:                        {int(fecal["StudyDayNo"].min())} – {int(fecal["StudyDayNo"].max())}')

✅ **Cleaning result:** 526 valid fecal samples from 37 participants. The `fecal.status = 0` rows have zero matching entries in BURST — no microbiome data exists for them, so nothing is lost.

## Missing Values Analysis

We check missing data rates across both metadata tables before any analysis.

In [ ]:
def plot_missing(df, title, ax, top_n=20):
    miss = (df.isnull().mean() * 100).sort_values(ascending=False).head(top_n)
    miss = miss[miss > 0]
    if miss.empty:
        ax.text(0.5, 0.5, 'No missing values', ha='center', va='center',
                transform=ax.transAxes, fontsize=12)
    else:
        ax.barh(miss.index[::-1], miss.values[::-1], color='#C44E52')
        ax.set_xlabel('Missing (%)')
        ax.axvline(5, color='orange', linestyle='--', alpha=0.7, label='5% threshold')
        ax.legend()
    ax.set_title(title)

fig, axes = plt.subplots(1, 2, figsize=(14, 6))
plot_missing(fecal, 'Sample Metadata – Missing Values', axes[0])
plot_missing(user_map, 'Participant Metadata – Missing Values', axes[1])
plt.tight_layout()
plt.show()

✅ **Missing data:** Core analysis columns (SampleID, UserName, StudyDayNo, Supplement) are complete. Higher missingness in participant metadata is concentrated in clinical measurements for participants who dropped out.

## Study Design Overview

We visualize cohort composition: supplement arms, gender, and age distribution.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 4))

supp = user_map['Supplement'].value_counts()
axes[0].bar(supp.index, supp.values, color=['#4C72B0','#DD8452'], edgecolor='white', width=0.5)
axes[0].set_title('Supplement Arms')
axes[0].set_ylabel('Participants')
for i, v in enumerate(supp.values):
    axes[0].text(i, v + 0.1, str(v), ha='center', fontweight='bold')

gender = user_map['Gender'].value_counts()
axes[1].pie(gender.values, labels=gender.index, autopct='%1.0f%%',
            colors=['#4C72B0','#DD8452'], startangle=90)
axes[1].set_title('Gender Distribution')

axes[2].hist(user_map['Age'].dropna(), bins=10, color='#55A868', edgecolor='white')
axes[2].axvline(user_map['Age'].mean(), color='red', linestyle='--',
                label=f"Mean: {user_map['Age'].mean():.1f} yr")
axes[2].set_title('Age Distribution')
axes[2].set_xlabel('Age (years)')
axes[2].legend()

plt.suptitle('Study Design – 37 Participants', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

✅ **Demographics:** Arms are balanced. Cohort skews female, age range 20–45, consistent with a healthy volunteer population.

We also verify that longitudinal sampling is consistent across participants and study days.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

per_part = fecal.groupby('UserName')['StudyDayNo'].count().sort_values(ascending=False)
axes[0].bar(range(len(per_part)), per_part.values, color='#4C72B0')
axes[0].axhline(per_part.mean(), color='red', linestyle='--',
                label=f'Mean: {per_part.mean():.1f}')
axes[0].set_title('Fecal Samples per Participant')
axes[0].set_xlabel('Participant (ranked)')
axes[0].set_ylabel('# Samples')
axes[0].legend()

per_day = fecal.groupby('StudyDayNo')['SampleID'].count()
axes[1].bar(per_day.index, per_day.values, color='#55A868')
axes[1].set_title('Samples per Study Day')
axes[1].set_xlabel('Study Day')
axes[1].set_ylabel('# Samples')

plt.suptitle('Longitudinal Sampling Overview', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

✅ **Coverage:** Sampling is consistent — most participants contributed samples across nearly all 17 days with minimal gaps.

## Microbiome Preprocessing

Steps:
1. Subset BURST columns to valid fecal samples
2. Normalize raw counts → relative abundance (% per sample)
3. Filter taxa with mean relative abundance < 0.01% (noise removal)

In [ ]:
shared = [s for s in fecal['SampleID'] if s in burst.columns]
burst_fecal    = burst[shared].copy()
burst_rel      = burst_fecal.div(burst_fecal.sum(axis=0), axis=1) * 100
burst_filtered = burst_rel[burst_rel.mean(axis=1) >= 0.01]

print(f'Fecal samples matched to BURST:  {len(shared)} / {len(fecal)}')
print(f'Taxa before filtering:           {len(burst_rel):,}')
print(f'Taxa after filtering (>=0.01%):  {len(burst_filtered)}')
print(f'Mean read coverage retained:     {burst_filtered.sum().mean():.1f}%')

✅ All 526 samples matched. 293 informative taxa retained from 4,583 (preserving ~99% of reads).

## Taxonomic Composition

We identify the 10 most abundant genera across all fecal samples.

In [ ]:
is_genus = (burst_filtered.index.str.contains(';g__') &
            ~burst_filtered.index.str.contains(';s__'))
burst_genus = burst_filtered[is_genus].copy()
burst_genus.index = burst_genus.index.map(
    lambda x: x.split(';g__')[-1].replace('_', ' '))

top10 = burst_genus.mean(axis=1).sort_values(ascending=False).head(10)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

colors10 = sns.color_palette('tab10', 10)
axes[0].barh(top10.index[::-1], top10.values[::-1], color=colors10)
axes[0].set_xlabel('Mean Relative Abundance (%)')
axes[0].set_title('Top 10 Genera – Mean Abundance')
for i, (v, n) in enumerate(zip(top10.values[::-1], top10.index[::-1])):
    axes[0].text(v + 0.1, i, f'{v:.1f}%', va='center', fontsize=8)

other  = 100 - top10.sum()
vals   = list(top10.values) + [other]
labels = list(top10.index)  + ['Other']
colors = list(colors10) + [(0.82, 0.82, 0.82)]
axes[1].pie(vals, labels=labels, autopct='%1.1f%%', colors=colors,
            textprops={'fontsize': 8}, startangle=90)
axes[1].set_title('Genus-level Composition')

plt.suptitle('Gut Microbiome – Taxonomic Composition', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

✅ A small number of genera (typically *Bacteroides*, *Faecalibacterium*, *Blautia*, *Ruminococcus*) dominate relative abundance. The large 'Other' slice reflects the high diversity of low-abundance taxa in a healthy gut.

## Alpha Diversity

We compute two within-sample diversity metrics:
- **Shannon entropy** — rewards both richness and evenness
- **Observed species** — count of species with relative abundance > 0

Computed at species level (not genus level) for accuracy.

In [ ]:
is_sp = (burst_filtered.index.str.contains(';s__') &
         ~burst_filtered.index.str.contains(';t__'))
burst_sp = burst_filtered[is_sp].copy()
burst_sp.index = burst_sp.index.map(
    lambda x: x.split(';s__')[-1].replace('_', ' '))

def shannon(col):
    p = col[col > 0] / 100
    return -(p * np.log(p)).sum()

alpha = pd.DataFrame({
    'SampleID': burst_sp.columns,
    'Shannon':  burst_sp.apply(shannon, axis=0).values,
    'Observed': (burst_sp > 0).sum(axis=0).values
})
alpha = alpha.merge(
    fecal[['SampleID','UserName','StudyDayNo','Supplement']],
    on='SampleID', how='left')

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for ax, col, lbl in zip(axes,
        ['Shannon','Observed'], ['Shannon Entropy','Observed Species']):
    ax.hist(alpha[col].dropna(), bins=25, color='#4C72B0', edgecolor='white')
    ax.axvline(alpha[col].mean(), color='red', linestyle='--',
               label=f'Mean: {alpha[col].mean():.2f}')
    ax.set_xlabel(lbl)
    ax.set_ylabel('# Samples')
    ax.set_title(f'Distribution of {lbl}')
    ax.legend()
plt.suptitle('Alpha Diversity – All Fecal Samples', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

✅ Both metrics are approximately normally distributed, indicating consistent microbiome complexity across the cohort.

### Alpha Diversity: MCT Oil vs. EVOO

We test whether the two supplement arms differ in gut microbiome diversity.

In [ ]:
palette = {'MCT': '#4C72B0', 'EVOO': '#DD8452'}
data_alpha = alpha[alpha['Supplement'].isin(['MCT','EVOO'])]

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
for ax, col, lbl in zip(axes,
        ['Shannon','Observed'], ['Shannon Entropy','Observed Species']):
    sns.boxplot(data=data_alpha, x='Supplement', y=col,
                palette=palette, ax=ax, width=0.5)
    sns.stripplot(data=data_alpha, x='Supplement', y=col,
                  palette=palette, ax=ax, alpha=0.25, jitter=True, size=3)
    g1 = data_alpha[data_alpha['Supplement']=='MCT'][col].dropna()
    g2 = data_alpha[data_alpha['Supplement']=='EVOO'][col].dropna()
    _, pval = mannwhitneyu(g1, g2, alternative='two-sided')
    ax.set_title(f'{lbl}\np (Mann-Whitney) = {pval:.3f}')
    ax.set_ylabel(lbl); ax.set_xlabel('')
plt.suptitle('Alpha Diversity: MCT Oil vs. EVOO', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

✅ **MCT vs. EVOO:** The p-value indicates whether the arms differ significantly in diversity. The p-value > 0.05 would suggest both interventions produce similar overall gut diversity.

### Alpha Diversity Over Study Days (Longitudinal)

Day-by-day mean diversity per arm reveals any time-dependent trends.

In [ ]:
by_day = data_alpha.groupby(['StudyDayNo','Supplement'])[['Shannon','Observed']].mean().reset_index()

fig, axes = plt.subplots(1, 2, figsize=(14, 4))
for ax, col, lbl in zip(axes,
        ['Shannon','Observed'], ['Shannon Entropy','Observed Species']):
    for supp, color in palette.items():
        d = by_day[by_day['Supplement']==supp].sort_values('StudyDayNo')
        ax.plot(d['StudyDayNo'], d[col], marker='o', label=supp,
                color=color, linewidth=2, markersize=4)
    ax.set_xlabel('Study Day'); ax.set_ylabel(f'Mean {lbl}')
    ax.set_title(f'{lbl} over Study Days'); ax.legend()
plt.suptitle('Longitudinal Alpha Diversity by Supplement Arm', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

✅ **Longitudinal diversity:** Diverging lines between arms would indicate a time-dependent dietary effect on gut microbiome diversity.

## Nutrition Analysis

Each participant kept a daily food diary. We analyze five key macronutrients: calories (KCAL), protein (PROT), total fat (TFAT), carbohydrates (CARB), and dietary fiber (FIBE).

In [ ]:
nutr_fecal = nutrition[nutrition['SampleID'].isin(fecal['SampleID'])].copy()
nutr_fecal = nutr_fecal.merge(
    fecal[['SampleID','UserName','StudyDayNo','Supplement']],
    on='SampleID', how='left')

macros = ['KCAL','PROT','TFAT','CARB','FIBE']
macro_labels = {'KCAL':'Energy (kcal)','PROT':'Protein (g)',
                'TFAT':'Total Fat (g)','CARB':'Carbs (g)','FIBE':'Fiber (g)'}

print(f'Nutrition records matched to valid samples: {len(nutr_fecal)}')
print(f'Participants with nutrition data:           {nutr_fecal["UserName"].nunique()}')
print()
for m in macros:
    v = nutr_fecal[m].dropna()
    print(f'{macro_labels[m]:<20}  mean={v.mean():.1f}  median={v.median():.1f}  std={v.std():.1f}')

✅ Nutrition data merged. Summary statistics show the typical daily macronutrient intake across all sample-days.

### Macronutrient Distributions

Histogram of each macronutrient across all sample-days.

In [ ]:
fig, axes = plt.subplots(1, 5, figsize=(18, 4))
for ax, m in zip(axes, macros):
    vals = nutr_fecal[m].dropna()
    ax.hist(vals, bins=25, color='#4C72B0', edgecolor='white')
    ax.axvline(vals.mean(), color='red', linestyle='--',
               label=f'Mean: {vals.mean():.0f}')
    ax.set_title(macro_labels[m])
    ax.legend(fontsize=8)
plt.suptitle('Macronutrient Distributions – All Sample-Days', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

✅ **Distributions:** Energy intake is approximately normal. Protein and fat show right-skewed distributions, typical of self-reported dietary data.

### Nutrition: MCT Oil vs. EVOO

We compare macronutrient intake between the two arms. Similar intake would confirm the arms had comparable diets and microbiome differences are due to the supplement itself.

In [ ]:
data_nutr = nutr_fecal[nutr_fecal['Supplement'].isin(['MCT','EVOO'])]

fig, axes = plt.subplots(1, 5, figsize=(18, 5))
for ax, m in zip(axes, macros):
    sns.boxplot(data=data_nutr, x='Supplement', y=m,
                palette=palette, ax=ax, width=0.5)
    sns.stripplot(data=data_nutr, x='Supplement', y=m,
                  palette=palette, ax=ax, alpha=0.2, jitter=True, size=2)
    g1 = data_nutr[data_nutr['Supplement']=='MCT'][m].dropna()
    g2 = data_nutr[data_nutr['Supplement']=='EVOO'][m].dropna()
    _, pval = mannwhitneyu(g1, g2, alternative='two-sided')
    ax.set_title(f'{macro_labels[m]}\np={pval:.3f}', fontsize=9)
    ax.set_xlabel('')
plt.suptitle('Macronutrients: MCT Oil vs. EVOO', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

✅ **Diet comparison:** p-values confirm whether the arms had statistically similar macronutrient intake, which is important for interpreting downstream microbiome results.

### Calorie & Fiber Intake Over Study Days

We track energy and fiber trends over time per arm.

In [ ]:
nutr_day = data_nutr.groupby(['StudyDayNo','Supplement'])[macros].mean().reset_index()

fig, axes = plt.subplots(1, 2, figsize=(14, 4))
for ax, m, lbl in zip(axes, ['KCAL','FIBE'], ['Energy (kcal)','Dietary Fiber (g)']):
    for supp, color in palette.items():
        d = nutr_day[nutr_day['Supplement']==supp].sort_values('StudyDayNo')
        ax.plot(d['StudyDayNo'], d[m], marker='o', label=supp,
                color=color, linewidth=2, markersize=4)
    ax.set_xlabel('Study Day'); ax.set_ylabel(f'Mean {lbl}')
    ax.set_title(f'{lbl} over Study Days'); ax.legend()
plt.suptitle('Longitudinal Nutrition: Energy & Fiber', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

✅ **Longitudinal diet:** Relatively stable intake over time reduces the risk of time-based confounding in the microbiome analysis.

## Food Group Analysis

We use the food taxonomy reference to classify individual foods into main groups (e.g. Grains, Meat, Vegetables, Dairy) and compare consumption between supplement arms.

In [ ]:
# Extract L1 food category from taxonomy hierarchy string
food_tax['L1'] = (food_tax['taxonomy']
                  .str.split(';').str[0]
                  .str.replace('L1_', '', regex=False)
                  .str.replace('_', ' ', regex=False))

# Map each food item (diet_fiber row) to its L1 category
food_to_l1 = food_tax.set_index('Main.food.description')['L1'].to_dict()

df_f = diet_fiber.copy()
df_f['L1'] = df_f.index.map(food_to_l1)
df_mapped = df_f.dropna(subset=['L1'])
print(f'Food items mapped to L1 category: {len(df_mapped)} / {len(df_f)}')

# Sum consumption by L1 group per sample
foodgrp = df_mapped.groupby('L1').sum()
valid_cols = [c for c in foodgrp.columns if c in fecal['SampleID'].values]
foodgrp = foodgrp[valid_cols]

top_groups = foodgrp.mean(axis=1).sort_values(ascending=False)
print('\nFood groups by mean daily consumption:')
for g, v in top_groups.items():
    print(f'  {g:<45} {v:.1f}')

In [ ]:
top_n = top_groups.head(9)
colors_grp = sns.color_palette('Set2', len(top_n))

fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# Overall consumption
axes[0].barh(top_n.index[::-1], top_n.values[::-1], color=colors_grp[::-1])
axes[0].set_xlabel('Mean Daily Consumption')
axes[0].set_title('Food Group Consumption – All Participants')

# MCT vs EVOO
supp_map = fecal.set_index('SampleID')['Supplement']
fg_T = foodgrp[valid_cols].T.copy()
fg_T['Supplement'] = fg_T.index.map(supp_map)
fg_means = fg_T.groupby('Supplement')[list(top_n.index)].mean().T

x = np.arange(len(fg_means))
w = 0.35
axes[1].barh(x + w/2, fg_means.get('MCT',  pd.Series(dtype=float)), w,
             label='MCT',  color='#4C72B0')
axes[1].barh(x - w/2, fg_means.get('EVOO', pd.Series(dtype=float)), w,
             label='EVOO', color='#DD8452')
axes[1].set_yticks(x)
axes[1].set_yticklabels(fg_means.index, fontsize=9)
axes[1].set_xlabel('Mean Daily Consumption')
axes[1].set_title('Food Groups: MCT vs. EVOO')
axes[1].legend()

plt.suptitle('Food Group Composition', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

✅ **Food groups:** The distribution reveals which categories dominate the cohort's diet. Comparing MCT vs. EVOO tests whether the arms had different food preferences that could confound results.

## Clinical Data – Blood Tests Before vs. After

Each participant had blood drawn at **Baseline** and **Final**. We assess whether the dietary intervention produced measurable clinical changes and whether changes differed between arms.

**Biomarkers:** Total Cholesterol, Triglycerides, HDL, LDL, Glucose, Insulin, HOMA-IR

In [ ]:
blood_cols = ['Cholesterol','Trigs','HDL','LDL','Glu','Ins','Homa.IR']
blood_lbls = {'Cholesterol':'Cholesterol','Trigs':'Triglycerides',
              'HDL':'HDL','LDL':'LDL','Glu':'Glucose',
              'Ins':'Insulin','Homa.IR':'HOMA-IR'}

complete = user_long[user_long['Study.Status']=='Complete'].copy()
print(f'Complete participants with both blood draws: {complete["UserName"].nunique()}')

fig, axes = plt.subplots(2, 4, figsize=(16, 8))
axes = axes.flatten()

for ax, col in zip(axes, blood_cols):
    base_df  = complete[complete['Blood.draw']=='Baseline'][['UserName',col]].dropna()
    final_df = complete[complete['Blood.draw']=='Final'][['UserName',col]].dropna()
    paired   = base_df.merge(final_df, on='UserName', suffixes=('_b','_f'))
    for _, row in paired.iterrows():
        ax.plot([0,1], [row[col+'_b'], row[col+'_f']],
                color='gray', alpha=0.3, linewidth=0.8)
    ax.plot([0,1], [paired[col+'_b'].mean(), paired[col+'_f'].mean()],
            color='red', linewidth=2.5, marker='o', markersize=7, label='Mean')
    ax.set_xticks([0,1]); ax.set_xticklabels(['Baseline','Final'])
    ax.set_title(blood_lbls[col]); ax.legend(fontsize=8)

axes[-1].axis('off')
plt.suptitle('Blood Biomarkers: Baseline → Final (Complete Participants)',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# Change = Final - Baseline per participant, compared between arms
complete_wide = complete.pivot(index='UserName', columns='Blood.draw', values=blood_cols)
complete_wide.columns = [f'{c}_{t}' for c, t in complete_wide.columns]
for col in blood_cols:
    complete_wide[f'{col}_change'] = (complete_wide.get(f'{col}_Final', np.nan) -
                                      complete_wide.get(f'{col}_Baseline', np.nan))
complete_wide = complete_wide.reset_index().merge(
    user_map[['UserName','Supplement']], on='UserName', how='left')
complete_wide = complete_wide[complete_wide['Supplement'].isin(['MCT','EVOO'])]

fig, axes = plt.subplots(2, 4, figsize=(16, 7))
axes = axes.flatten()
for ax, col in zip(axes, blood_cols):
    cname = f'{col}_change'
    sns.boxplot(data=complete_wide, x='Supplement', y=cname,
                palette=palette, ax=ax, width=0.5)
    sns.stripplot(data=complete_wide, x='Supplement', y=cname,
                  palette=palette, ax=ax, alpha=0.5, size=5)
    ax.axhline(0, color='red', linestyle='--', alpha=0.6)
    g1 = complete_wide[complete_wide['Supplement']=='MCT'][cname].dropna()
    g2 = complete_wide[complete_wide['Supplement']=='EVOO'][cname].dropna()
    if len(g1) > 1 and len(g2) > 1:
        _, pval = mannwhitneyu(g1, g2, alternative='two-sided')
        ax.set_title(f'Δ{blood_lbls[col]}\np={pval:.3f}', fontsize=9)
    ax.set_xlabel(''); ax.set_ylabel('Change (Final − Baseline)')
axes[-1].axis('off')
plt.suptitle('Blood Biomarker Changes: MCT vs. EVOO',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

✅ **Clinical outcomes:** Paired lines show individual Baseline→Final trajectories with the mean in red. The second panel tests whether MCT and EVOO produced different magnitudes of change (dashed red = no change).

## Diet ↔ Microbiome Correlations

We link each sample's alpha diversity to the nutritional content recorded for the same day. This reveals which dietary factors are most associated with gut microbiome diversity.

In [ ]:
extra_nuts = ['SUGR','SODI','CALC','IRON']
corr_data = alpha[['SampleID','Shannon','Observed']].merge(
    nutr_fecal[['SampleID'] + macros + extra_nuts],
    on='SampleID', how='inner')

nut_display = {'KCAL':'Energy','PROT':'Protein','TFAT':'Fat',
               'CARB':'Carbs','FIBE':'Fiber',
               'SUGR':'Sugars','SODI':'Sodium','CALC':'Calcium','IRON':'Iron'}
all_nuts = macros + extra_nuts

corr_matrix = corr_data[['Shannon','Observed'] + all_nuts].corr()
rename = {'Shannon':'Shannon Entropy','Observed':'Observed Species'}
rename.update(nut_display)
corr_display = corr_matrix.rename(index=rename, columns=rename)

fig, ax = plt.subplots(figsize=(11, 9))
sns.heatmap(corr_display, annot=True, fmt='.2f', cmap='RdBu_r',
            center=0, vmin=-1, vmax=1, linewidths=0.5, ax=ax, square=True,
            annot_kws={'size': 8})
ax.set_title('Correlation Matrix: Alpha Diversity × Nutrient Intake',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# Bar chart: nutrients ranked by correlation with Shannon diversity
shannon_corr = (corr_data[all_nuts + ['Shannon']]
                .corr()['Shannon'].drop('Shannon')
                .rename(nut_display)
                .sort_values())

colors_bar = ['#C44E52' if v < 0 else '#4C72B0' for v in shannon_corr.values]
fig, ax = plt.subplots(figsize=(8, 5))
ax.barh(shannon_corr.index, shannon_corr.values, color=colors_bar)
ax.axvline(0, color='black', linewidth=0.8)
ax.set_xlabel('Pearson Correlation with Shannon Entropy')
ax.set_title('Which Nutrients Are Most Correlated with Gut Diversity?',
             fontweight='bold')
plt.tight_layout()
plt.show()

print('Nutrient correlations with Shannon (ranked by |r|):')
ranked = shannon_corr.reindex(shannon_corr.abs().sort_values(ascending=False).index)
for k, v in ranked.items():
    print(f'  {k:<12}  r = {v:.3f}')

✅ **Diet–Microbiome link:**
- **Blue bars (positive r):** higher intake → more diverse gut microbiome
- **Red bars (negative r):** higher intake → less diverse gut microbiome

Dietary fiber is typically expected to positively correlate with diversity, as it feeds beneficial bacteria. High sugar/sodium intake is often associated with reduced diversity.